In [0]:
from pyspark.sql.functions import sum, countDistinct, round, col, avg, date_format

In [0]:
silver_df = spark.table("adbrag.project4_schema.sales_silver")

In [0]:
spark.table("adbrag.project4_schema.sales_silver") \
    .groupBy("SalesOrderNumber", "SalesOrderLineNumber") \
    .count() \
    .filter("count > 1") \
    .display()

In [0]:
spark.table("adbrag.project4_schema.sales_silver") \
    .selectExpr(
        "count(*) as total_rows",
        "count(SalesOrderNumber) as non_null_orders",
        "count(ProductName) as non_null_products"
    ).display()

In [0]:
silver_total = spark.table("adbrag.project4_schema.sales_silver") \
    .select(sum(col("OrderQuantity") * col("UnitPrice")).alias("total"))

gold_total = spark.table("adbrag.project4_schema.gold_yearly_summary") \
    .select(sum("TotalSalesAmount").alias("total"))

silver_total.display()
gold_total.display()

In [0]:

# 1. Product-Year Sales
gold_product_year = (
    silver_df.groupBy("SourceYear", "ProductName")
    .agg(
        countDistinct("SalesOrderNumber").alias("TotalOrders"),
        sum("OrderQuantity").alias("TotalQuantity"),
        round(sum(col("OrderQuantity") * col("UnitPrice")), 2).alias("TotalSalesAmount"),
        round(sum("TaxAmount"), 2).alias("TotalTaxAmount")
    )
)
gold_product_year.write.format("delta").mode("overwrite").saveAsTable("adbrag.project4_schema.gold_product_year_sales")

# 2. Customer Sales
gold_customer_sales = (
    silver_df.groupBy("CustomerName", "EmailAddress")
    .agg(
        countDistinct("SalesOrderNumber").alias("TotalOrders"),
        sum("OrderQuantity").alias("TotalQuantity"),
        round(sum(col("OrderQuantity") * col("UnitPrice")), 2).alias("TotalSalesAmount"),
        round(sum("TaxAmount"), 2).alias("TotalTaxAmount")
    )
)
gold_customer_sales.write.format("delta").mode("overwrite").saveAsTable("adbrag.project4_schema.gold_customer_sales")

# 3. Yearly Summary
gold_yearly_summary = (
    silver_df.groupBy("SourceYear")
    .agg(
        countDistinct("SalesOrderNumber").alias("TotalOrders"),
        sum("OrderQuantity").alias("TotalQuantity"),
        round(sum(col("OrderQuantity") * col("UnitPrice")), 2).alias("TotalSalesAmount"),
        round(sum("TaxAmount"), 2).alias("TotalTaxAmount"),
        round(avg("UnitPrice"), 2).alias("AverageUnitPrice")
    )
)
gold_yearly_summary.write.format("delta").mode("overwrite").saveAsTable("adbrag.project4_schema.gold_yearly_summary")

# 4. Monthly Sales
gold_monthly_sales = (
    silver_df.withColumn("OrderMonth", date_format(col("OrderDate"), "yyyy-MM"))
    .groupBy("OrderMonth")
    .agg(
        countDistinct("SalesOrderNumber").alias("TotalOrders"),
        sum("OrderQuantity").alias("TotalQuantity"),
        round(sum(col("OrderQuantity") * col("UnitPrice")), 2).alias("TotalSalesAmount"),
        round(sum("TaxAmount"), 2).alias("TotalTaxAmount")
    )
)
gold_monthly_sales.write.format("delta").mode("overwrite").saveAsTable("adbrag.project4_schema.gold_monthly_sales")

print("All gold aggregations created successfully.")

In [0]:
spark.sql("SHOW TABLES IN adbrag.project4_schema").display()

In [0]:
df = spark.table("adbrag.project4_schema.gold_product_year_sales")
df.display()

In [0]:
df = spark.table("adbrag.project4_schema.gold_customer_sales")
df.display()

In [0]:
df = spark.table("adbrag.project4_schema.gold_yearly_summary")
df.display()

In [0]:
df = spark.table("adbrag.project4_schema.gold_monthly_sales")
df.orderBy("OrderMonth").display()